# Enterprise Customer Support Triage and Resolution Desk
### Course: CSE476 – Agentic AI and Intelligent Automation (CA2 - Multi-Agent System)
**Selected Topic:** Group 3 – Enterprise Customer Support Triage and Resolution Desk

---
### Multi-Agent Architecture Overview
This notebook demonstrates a collaborative **Router-to-Specialists** multi-agent system built using:
- **Framework:** Microsoft Semantic Kernel (v1.44.1) & Python
- **LLM / Provider:** Groq (`openai/gpt-oss-20b`)
- **6 Mandatory Agents:**
  1. **Classifier Agent:** Categorizes into billing, technical, account, other with confidence & entity extraction.
  2. **Billing Specialist:** Resolves refund and invoice queries using genuine Python billing rules.
  3. **Technical Specialist:** Resolves errors and how-to queries using local support knowledge base RAG.
  4. **Account Specialist:** Resolves authentication and lockout queries using Python security tools.
  5. **Escalation Router:** Evaluates escalation criteria (low confidence, lack of evidence, unresolved issues) and generates human briefings.
  6. **Reviewer Agent:** Assesses response tone, completeness, and compliance; enforces revision loops up to max retries.
- **Shared Infrastructure:** Shared Blackboard, Append-only Event Log, Session Memory, and Local Vector Store.

In [1]:
# Step 1: System Imports and Path Setup
import os
import sys
import json
from pathlib import Path

# Ensure project root is on sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from config import GROQ_MODEL, PROVIDER, CONFIDENCE_THRESHOLD, SIMILARITY_THRESHOLD
from memory.session_memory import SessionMemory
from orchestration.workflow import SupportDeskOrchestrator
from rag.retriever import LocalVectorRetriever

print(f"[Environment Ready] Provider: {PROVIDER} | Model: {GROQ_MODEL}")
print(f"[Configuration] Confidence Threshold: {CONFIDENCE_THRESHOLD} | Similarity Threshold: {SIMILARITY_THRESHOLD}")

[Environment Ready] Provider: groq | Model: openai/gpt-oss-20b
[Configuration] Confidence Threshold: 0.65 | Similarity Threshold: 0.25


## Scenario 1: Normal Billing Issue (Payment Deducted, Order Unconfirmed)
**Customer:** *"My payment was deducted from my account, but my order was not confirmed. I have already waited for two hours. Order ORD-9912, transaction TXN-5544, amount $49.99."*

**Workflow Expected:**
1. Classifier identifies category `billing` with high confidence and extracts entities (`ORD-9912`, `TXN-5544`, `$49.99`).
2. Main Router dispatches to `BillingSpecialist`.
3. Billing Specialist invokes `verify_invoice_and_payment_status()` and `check_refund_eligibility()`.
4. Reviewer Agent checks response quality, tone, and completeness, approving the response.
5. Blackboard and Event Log record complete execution.

In [2]:
session_memory = SessionMemory()
orchestrator = SupportDeskOrchestrator(session_memory=session_memory)

ticket_1_msg = "My payment was deducted from my account, but my order was not confirmed. I have already waited for two hours. Order ORD-9912, transaction TXN-5544, amount $49.99."
result_1 = orchestrator.process_ticket(customer_message=ticket_1_msg, customer_id="CUST-1001")

print("=== TICKET 1 RESULT ===")
print(f"Ticket ID: {result_1['ticket_id']}")
print(f"Final Status: {result_1['status']}")
snap_1 = result_1['blackboard_snapshot']
print(f"Category: {snap_1['classification']['category']} (Confidence: {snap_1['classification']['confidence']})")
print(f"Assigned Specialist: {snap_1['routing']['target_specialist']}")
print(f"Specialist Resolution: {snap_1['specialist_output']['resolution']}")
print(f"Reviewer Approved: {snap_1['review']['approved']} (Tone Score: {snap_1['review']['tone_score']})")
print("\n--- CUSTOMER RESPONSE ---")
print(result_1['final_response'])

=== TICKET 1 RESULT ===
Ticket ID: TICKET-5DBC80
Final Status: RESOLVED
Category: billing (Confidence: 0.88)
Assigned Specialist: billing_specialist
Specialist Resolution: Payment Status: CAPTURED (PAYMENT_SUCCESS_WEBHOOK_DELAYED). Refund Evaluation: Transaction 5544 is within the 30-day money-back guarantee window (1 days elapsed). Eligible for 100% full refund to original payment source.
Reviewer Approved: True (Tone Score: 1.0)

--- CUSTOMER RESPONSE ---
Hello, thank you for reaching out to Enterprise Support regarding order was.

We have verified your payment details (Transaction: 5544). Our records show that your payment of $49.99 was successfully captured at the gateway level. Due to a brief synchronization delay with our order management system, your confirmation was held in queue. We have manually initiated the confirmation sync, and your order confirmation email will arrive within the hour.

Under our REF-POL-2026 terms, your transaction is protected by our full satisfaction g

## Scenario 2: Technical Support Issue with RAG Grounding
**Customer:** *"The web console is throwing an ERR-1001 gateway request timeout error when querying the analytics dashboard."*

**Workflow Expected:**
1. Classifier identifies category `technical` and extracts error code `ERR-1001`.
2. Main Router dispatches to `TechnicalSpecialist`.
3. Technical Specialist queries the local Knowledge Base via semantic RAG and retrieves official diagnostic remediation.
4. Cites exact source documentation (`[error_codes.txt#ERR-1001]`).
5. Reviewer Agent validates citation presence and approves.

In [3]:
ticket_2_msg = "The web console is throwing an ERR-1001 gateway request timeout error when querying the analytics dashboard."
result_2 = orchestrator.process_ticket(customer_message=ticket_2_msg, customer_id="CUST-1002")

print("=== TICKET 2 RESULT ===")
print(f"Ticket ID: {result_2['ticket_id']}")
print(f"Final Status: {result_2['status']}")
snap_2 = result_2['blackboard_snapshot']
print(f"Category: {snap_2['classification']['category']} (Confidence: {snap_2['classification']['confidence']})")
print(f"Assigned Specialist: {snap_2['routing']['target_specialist']}")
print(f"Evidence / Citations: {snap_2['specialist_output']['evidence']}")
print("\n--- CUSTOMER RESPONSE ---")
print(result_2['final_response'])

=== TICKET 2 RESULT ===
Ticket ID: TICKET-4B37FA
Final Status: RESOLVED
Category: technical (Confidence: 0.98)
Assigned Specialist: technical_specialist
Evidence / Citations: ['[error_codes.txt#ERR-1001]']

--- CUSTOMER RESPONSE ---
Hello, thank you for contacting Enterprise Technical Support.

We have identified the root cause for ERR-1001 (Gateway Request Timeout). This issue occurs because: Upstream microservice timed out after 30 seconds due to backlog..

Recommended Resolution Steps:
1. Instruct user to wait 60 seconds and retry with exponential backoff. Check system status page.
2. Reference Documentation: [error_codes.txt#ERR-1001].

If the problem persists after following these steps, please let us know so our team can investigate your environment.


## Scenario 3: Account Security and Lockout
**Customer:** *"I am locked out of my account ACC-8822 after entering the wrong password. Please help me reset access."*

**Workflow Expected:**
1. Classifier identifies `account` category and extracts `ACC-8822`.
2. Main Router dispatches to `AccountSpecialist`.
3. Account Specialist verifies security status and generates a secure, timed reset token (`RST-...`).
4. Reviewer approves safe, compliant customer response.

In [4]:
ticket_3_msg = "I am locked out of my account ACC-8822 after entering the wrong password. Please help me reset access."
result_3 = orchestrator.process_ticket(customer_message=ticket_3_msg, customer_id="CUST-1003")

print("=== TICKET 3 RESULT ===")
print(f"Ticket ID: {result_3['ticket_id']}")
snap_3 = result_3['blackboard_snapshot']
print(f"Category: {snap_3['classification']['category']}")
print(f"Reset Token Issued: {snap_3['specialist_output']['metadata'].get('reset_token')}")
print("\n--- CUSTOMER RESPONSE ---")
print(result_3['final_response'])

=== TICKET 3 RESULT ===
Ticket ID: TICKET-74DD85
Category: account
Reset Token Issued: RST-A46C9563EB28

--- CUSTOMER RESPONSE ---
Hello, thank you for reaching out to Enterprise Account Support.

We have verified the security status for account 8822. Account is in healthy standing.

Next Steps to Access Your Account:
1. A one-time secure password reset link (Token: RST-A46C9563EB28) has been dispatched to EMAIL. Link expires strictly in 15 minutes. For security, all other active sessions have been terminated.
2. Once reset, ensure Two-Factor Authentication (2FA) is re-confirmed in your Profile Settings.

If you did not request this recovery or encounter further login friction, please notify our team immediately.


## Scenario 4: Insufficient Knowledge Base Information -> Human Escalation
**Customer:** *"I have a problem with a proprietary holographic rendering firmware that is not covered in your support documentation."*

**Workflow Expected:**
1. Classifier identifies `technical` category.
2. Technical Specialist executes RAG vector retrieval; cosine similarity is below threshold.
3. Specialist explicitly identifies **insufficient evidence** and refuses to hallucinate an answer.
4. Handoff to `EscalationRouter`, which routes ticket to **Tier 2 Technical Support Engineering** and crafts a formal internal briefing note.

In [5]:
ticket_4_msg = "I have a problem with a proprietary holographic rendering firmware that is not covered in your support documentation."
result_4 = orchestrator.process_ticket(customer_message=ticket_4_msg, customer_id="CUST-1004")

print("=== TICKET 4 RESULT (INSUFFICIENT RAG EVIDENCE) ===")
snap_4 = result_4['blackboard_snapshot']
print(f"Final Status: {result_4['status']}")
print(f"Escalated: {snap_4['escalation']['is_escalated']}")
print(f"Target Queue: {snap_4['escalation']['target_tier']}")
print(f"Priority: {snap_4['escalation']['priority']}")
print(f"Escalation Reason: {snap_4['escalation']['reason']}")
print("\n--- INTERNAL HANDOFF NOTE ---")
print(snap_4['escalation']['handoff_note'])
print("\n--- CUSTOMER EXPLANATION ---")
print(result_4['final_response'])

=== TICKET 4 RESULT (INSUFFICIENT RAG EVIDENCE) ===
Final Status: ESCALATED
Escalated: True
Target Queue: Tier 2 Technical Support Engineering
Priority: HIGH
Escalation Reason: RAG search returned insufficient evidence in local knowledge base.

--- INTERNAL HANDOFF NOTE ---
=== HUMAN ESCALATION BRIEFING ===
Ticket ID: TICKET-1FA285
Assigned Queue: Tier 2 Technical Support Engineering
Priority Level: HIGH
Primary Escalation Reason: RAG search returned insufficient evidence in local knowledge base.
Automated Specialist Diagnostics:
RAG search returned insufficient evidence in local knowledge base.
Action Required: Human agent manual intervention and direct customer follow-up.

--- CUSTOMER EXPLANATION ---
Hello, thank you for contacting Enterprise Support.

We have reviewed your request regarding your technical inquiry. To ensure you receive the highest quality assistance, your ticket has been prioritized and routed to our Tier 2 Technical Support Engineering (Priority: HIGH).

Reason fo

## Scenario 5: Ambiguous / Low-Confidence / 'Other' Category Ticket
**Customer:** *"Can you arrange a lawn mowing service for my suburban property tomorrow morning?"*

**Workflow Expected:**
1. Classifier identifies query as outside known domains -> category `'other'` with low confidence (< 0.65).
2. Main Router recognizes low confidence / unhandled domain and routes directly to `EscalationRouter`.
3. Escalation Router creates a manual review ticket in the General Operations queue.

In [6]:
ticket_5_msg = "Can you arrange a lawn mowing service for my suburban property tomorrow morning?"
result_5 = orchestrator.process_ticket(customer_message=ticket_5_msg, customer_id="CUST-1005")

print("=== TICKET 5 RESULT (OUT-OF-DOMAIN / OTHER) ===")
snap_5 = result_5['blackboard_snapshot']
print(f"Category: {snap_5['classification']['category']} (Confidence: {snap_5['classification']['confidence']})")
print(f"Routing Target: {snap_5['routing']['target_specialist']}")
print(f"Escalated: {snap_5['escalation']['is_escalated']}")
print(f"Target Queue: {snap_5['escalation']['target_tier']}")
print("\n--- CUSTOMER RESPONSE ---")
print(result_5['final_response'])

=== TICKET 5 RESULT (OUT-OF-DOMAIN / OTHER) ===
Category: other (Confidence: 0.45)
Routing Target: escalation_router
Escalated: True
Target Queue: Tier 2 Technical Support Engineering

--- CUSTOMER RESPONSE ---
Hello, thank you for contacting Enterprise Support.

We have reviewed your request regarding your other inquiry. To ensure you receive the highest quality assistance, your ticket has been prioritized and routed to our Tier 2 Technical Support Engineering (Priority: HIGH).

Reason for specialist transfer: Classification confidence (0.45) is below operational threshold (0.65).

A member of our team is reviewing your case history and will follow up with you directly.


## Scenario 6: Multi-Turn Conversation & Session Memory Retention
A returning customer asks a follow-up inquiry about their earlier ticket.

**Turn 1:** Customer submitted payment issue earlier (`CUST-1001` in Scenario 1).
**Turn 2:** Customer asks: *"Can you tell me which specialist handled my earlier payment issue and what was the resolution?"*

The system accesses stored `SessionMemory` to retrieve the previous ticket history and answer authoritatively.

In [7]:
# Inspect stored session memory for customer CUST-1001
customer_history = session_memory.get_customer_history("CUST-1001")
print(f"Customer CUST-1001 Stored Interaction Count: {len(customer_history)}")
print(f"Previous Ticket Summary: {customer_history[0]}")

# Context block assembled for follow-up query
context_block = session_memory.build_history_context("CUST-1001")
print("\n=== RECALLED SESSION CONTEXT ===")
print(context_block)

Customer CUST-1001 Stored Interaction Count: 1
Previous Ticket Summary: {'ticket_id': 'TICKET-5DBC80', 'created_at': '2026-11-20T11:10:47.551382+00:00', 'status': 'RESOLVED', 'category': 'billing', 'target_specialist': 'billing_specialist', 'final_response': 'Hello, thank you for reaching out to Enterprise Support regarding order was.\n\nWe have verified your payment details (Transaction: 5544). Our records show that your payment of $49.99 was successfully captured at the gateway level. Due to a brief synchronization delay with our order management system, your confirmation was held in queue. We have manually initiated the confirmation sync, and your order confirmation email will arrive within the hour.\n\nUnder our REF-POL-2026 terms, your transaction is protected by our full satisfaction guarantee. Please let us know if we can assist you with anything further.', 'is_escalated': False}

=== RECALLED SESSION CONTEXT ===
Customer CUST-1001 Previous Interaction History:
- Ticket [TICKET-

## Auditable Event Log Inspection
Every stage across every ticket is recorded in an **immutable, append-only Event Log** providing complete traceability.

In [8]:
# Display event log for Ticket 1
print(f"Auditable Event Log for Ticket 1 ({result_1['ticket_id']}):")
for evt in result_1['event_log']:
    print(f"  [{evt['timestamp'][:19]}] {evt['event_id']} | Agent: {evt['agent']:<20} | Action: {evt['action']:<25} | Status: {evt['status']}")

Auditable Event Log for Ticket 1 (TICKET-5DBC80):
  [2026-11-20T11:10:47] EVT-0001 | Agent: Orchestrator         | Action: TicketReceived            | Status: SUCCESS
  [2026-11-20T11:10:47] EVT-0002 | Agent: ClassifierAgent      | Action: TicketClassified          | Status: SUCCESS
  [2026-11-20T11:10:47] EVT-0003 | Agent: MainRouter           | Action: RoutingDecisionMade       | Status: SUCCESS
  [2026-11-20T11:10:47] EVT-0004 | Agent: BillingSpecialist    | Action: BillingRulesEvaluated     | Status: SUCCESS
  [2026-11-20T11:10:47] EVT-0005 | Agent: ReviewerAgent        | Action: ResponseApproved          | Status: SUCCESS
  [2026-11-20T11:10:47] EVT-0006 | Agent: Orchestrator         | Action: WorkflowCompleted         | Status: SUCCESS
